In [0]:
max_version = max(map(lambda v: v.name, dbutils.fs.ls('/pipelines/4ee489c5-1b41-44b3-bef4-2ea2ad7c4f0a/checkpoints/bronze/')))
print(max_version)
dbutils.widgets.remove("latest_checkpoint_version")
dbutils.widgets.text("latest_checkpoint_version",max_version)

1/


In [0]:
%sql
select b.source_metadata.file_name, a.create_time file_created, 
a.discovery_time discovered_by_autoloader, b._metadata.file_modification_time written_to_bronze, s._metadata.file_modification_time written_to_silver, g._metadata.file_modification_time written_to_gold, count(*) row_count
from cloud_files_state('/pipelines/4ee489c5-1b41-44b3-bef4-2ea2ad7c4f0a/checkpoints/bronze/${latest_checkpoint_version}') a
join hive_metastore.hops.bronze b on regexp_extract(a.path, '.*/([^/]*)$') = regexp_extract(b.source_metadata.file_path, '.*/([^/]*)$')
join hive_metastore.hops.silver s on b.tpep_dropoff_datetime=s.tpep_dropoff_datetime and b.tpep_pickup_datetime=s.tpep_pickup_datetime
join hive_metastore.hops.gold g on b.tpep_dropoff_datetime=g.tpep_dropoff_datetime and b.tpep_pickup_datetime=g.tpep_pickup_datetime
group by 1,2,3,4,5,6
order by 1,2,3

file_name,file_created,discovered_by_autoloader,written_to_bronze,written_to_silver,written_to_gold,row_count
part-00000-tid-588438459401112141-12ecdaa8-7679-44c5-8983-424335155e87-2086-1-c000.json,2024-04-05T18:03:28.000Z,2025-01-17T16:49:18.255Z,2025-01-17T16:49:22.000Z,2025-01-17T16:49:31.000Z,2025-01-17T16:49:37.000Z,3656
part-00001-tid-588438459401112141-12ecdaa8-7679-44c5-8983-424335155e87-2087-1-c000.json,2024-04-05T18:03:28.000Z,2025-01-17T16:49:18.491Z,2025-01-17T16:49:22.000Z,2025-01-17T16:49:31.000Z,2025-01-17T16:49:37.000Z,3655
part-00002-tid-588438459401112141-12ecdaa8-7679-44c5-8983-424335155e87-2088-1-c000.json,2024-04-05T18:03:28.000Z,2025-01-17T16:49:18.491Z,2025-01-17T16:49:22.000Z,2025-01-17T16:49:31.000Z,2025-01-17T16:49:37.000Z,3656
part-00003-tid-588438459401112141-12ecdaa8-7679-44c5-8983-424335155e87-2089-1-c000.json,2024-04-05T18:03:28.000Z,2025-01-17T16:49:18.491Z,2025-01-17T16:49:22.000Z,2025-01-17T16:49:31.000Z,2025-01-17T16:49:37.000Z,3655
part-00004-tid-588438459401112141-12ecdaa8-7679-44c5-8983-424335155e87-2090-1-c000.json,2024-04-05T18:03:28.000Z,2025-01-17T16:49:18.491Z,2025-01-17T16:49:22.000Z,2025-01-17T16:49:31.000Z,2025-01-17T16:49:37.000Z,3655
part-00005-tid-588438459401112141-12ecdaa8-7679-44c5-8983-424335155e87-2091-1-c000.json,2024-04-05T18:03:28.000Z,2025-01-17T16:49:18.492Z,2025-01-17T16:49:22.000Z,2025-01-17T16:49:31.000Z,2025-01-17T16:49:37.000Z,3655


In [0]:
%sql
select 
timestampdiff(second, discovered_by_autoloader, written_to_bronze) AS autoloader_to_bronze,
timestampdiff(second, written_to_bronze, written_to_silver) AS bronze_to_silver,
timestampdiff(second, written_to_silver, written_to_gold) AS silver_to_gold
from (
  select b.source_metadata.file_name, a.create_time file_created, 
  a.discovery_time discovered_by_autoloader, b._metadata.file_modification_time written_to_bronze, s._metadata.file_modification_time written_to_silver, g._metadata.file_modification_time written_to_gold, count(*) row_count
  from cloud_files_state('/pipelines/4ee489c5-1b41-44b3-bef4-2ea2ad7c4f0a/checkpoints/bronze/${latest_checkpoint_version}') a
  join hive_metastore.hops.bronze b on regexp_extract(a.path, '.*/([^/]*)$') = regexp_extract(b.source_metadata.file_path, '.*/([^/]*)$')
  join hive_metastore.hops.silver s on b.tpep_dropoff_datetime=s.tpep_dropoff_datetime and b.tpep_pickup_datetime=s.tpep_pickup_datetime
  join hive_metastore.hops.gold g on b.tpep_dropoff_datetime=g.tpep_dropoff_datetime and b.tpep_pickup_datetime=g.tpep_pickup_datetime
  group by 1,2,3,4,5,6
  order by 1,2,3
)

autoloader_to_bronze,bronze_to_silver,silver_to_gold
3,9,6
3,9,6
3,9,6
3,9,6
3,9,6
3,9,6


Databricks visualization. Run in Databricks to view.